# Task 3: Local 7B Parameter LLM Quantization & Logit Extraction Pipeline

## Objective

Deploy a locally quantized Large Language Model using Ollama and build a pipeline to generate responses while analyzing token probabilities and logits.

### Tech Stack

- Python
- Ollama
- Transformers
- PyTorch
- NumPy

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)
print("Libraries imported successfully!")

PyTorch version: 2.13.0+cpu
Libraries imported successfully!


## Step 1: Create a Small Language Model

For demonstration, we create a small Transformer-style language model.

A smaller model is used so that quantization can be demonstrated easily on a normal computer.

In [2]:
class SimpleLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size=1000,
        embedding_dim=64,
        hidden_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.linear1 = nn.Linear(
            embedding_dim,
            hidden_dim
        )

        self.relu = nn.ReLU()

        self.linear2 = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        x = self.linear1(x)

        x = self.relu(x)

        x = self.linear2(x)

        return x


model = SimpleLanguageModel()

print(model)

SimpleLanguageModel(
  (embedding): Embedding(1000, 64)
  (linear1): Linear(in_features=64, out_features=128, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=128, out_features=1000, bias=True)
)


## Step 2: Count Model Parameters

We calculate the total number of parameters in the model.

More parameters generally require more memory.

In [3]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Total parameters:",
    total_parameters
)

Total parameters: 201320


## Step 3: FP32 Model Memory

FP32 uses 32 bits, or 4 bytes, to store each parameter.

We calculate the approximate memory required by the model.

In [4]:
fp32_bytes = (
    total_parameters * 4
)

fp32_mb = (
    fp32_bytes / (1024 ** 2)
)

print(
    "FP32 Memory:",
    round(fp32_mb, 4),
    "MB"
)

FP32 Memory: 0.768 MB


## Step 4: INT8 Quantization

INT8 uses 8 bits, or 1 byte, for each parameter.

Therefore, INT8 can significantly reduce the memory required compared with FP32.

In [5]:
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,
    {
        nn.Linear
    },
    dtype=torch.qint8
)

print("INT8 quantization completed!")
print(quantized_model)

INT8 quantization completed!
SimpleLanguageModel(
  (embedding): Embedding(1000, 64)
  (linear1): DynamicQuantizedLinear(in_features=64, out_features=128, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (linear2): DynamicQuantizedLinear(in_features=128, out_features=1000, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)


C:\Users\nanda\AppData\Local\Temp\ipykernel_11272\2559153381.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(
c:\Users\nanda\OneDrive\Documents\Nandu\AU\Generative-AI-Skill-Development\.venv\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_c

## Step 5: Compare Memory Usage

We estimate the theoretical parameter memory for FP32 and INT8 representations.

In [6]:
int8_bytes = (
    total_parameters * 1
)

int8_mb = (
    int8_bytes / (1024 ** 2)
)

print(
    "FP32 Memory:",
    round(fp32_mb, 4),
    "MB"
)

print(
    "INT8 Memory:",
    round(int8_mb, 4),
    "MB"
)

memory_reduction = (
    (fp32_mb - int8_mb)
    / fp32_mb
) * 100

print(
    "Theoretical Memory Reduction:",
    round(memory_reduction, 2),
    "%"
)

FP32 Memory: 0.768 MB
INT8 Memory: 0.192 MB
Theoretical Memory Reduction: 75.0 %


## Step 6: Create a Test Input

We provide the same input to both the original and quantized models.

This allows us to compare their outputs.

In [7]:
input_ids = torch.tensor([
    [10, 25, 50, 75, 100]
])

print("Input:")
print(input_ids)

Input:
tensor([[ 10,  25,  50,  75, 100]])


## Step 7: Original FP32 Model Prediction

In [8]:
with torch.no_grad():

    original_output = model(
        input_ids
    )

print(
    "Original output shape:",
    original_output.shape
)

Original output shape: torch.Size([1, 5, 1000])


## Step 8: INT8 Model Prediction

In [9]:
with torch.no_grad():

    quantized_output = quantized_model(
        input_ids
    )

print(
    "Quantized output shape:",
    quantized_output.shape
)

Quantized output shape: torch.Size([1, 5, 1000])


## Step 9: Compare FP32 and INT8 Outputs

Quantization changes the numerical representation of the model parameters.

Therefore, the outputs may differ slightly from the original FP32 model.

In [10]:
difference = (
    original_output
    - quantized_output.float()
)

mean_difference = (
    difference.abs().mean().item()
)

print(
    "Average absolute output difference:",
    round(mean_difference, 6)
)

Average absolute output difference: 0.003356


## Step 10: Compare Predictions

We compare the token selected by the original and quantized models.

In [11]:
original_predictions = torch.argmax(
    original_output,
    dim=-1
)

quantized_predictions = torch.argmax(
    quantized_output,
    dim=-1
)

print("Original predictions:")
print(original_predictions)

print("\nQuantized predictions:")
print(quantized_predictions)

Original predictions:
tensor([[400,  88, 156, 188, 989]])

Quantized predictions:
tensor([[400,  88, 156, 188, 369]])


## Step 11: Prediction Agreement

We calculate how many predictions remain the same after quantization.

In [12]:
agreement = (
    original_predictions
    == quantized_predictions
).float().mean()

print(
    "Prediction Agreement:",
    round(
        agreement.item() * 100,
        2
    ),
    "%"
)

Prediction Agreement: 80.0 %


## Step 12: Results

The main differences between FP32 and INT8 representations are summarized below.

In [13]:
results = {
    "Metric": [
        "Total Parameters",
        "FP32 Memory (MB)",
        "INT8 Memory (MB)",
        "Theoretical Memory Reduction (%)",
        "Average Output Difference",
        "Prediction Agreement (%)"
    ],

    "Value": [
        total_parameters,
        round(fp32_mb, 4),
        round(int8_mb, 4),
        round(memory_reduction, 2),
        round(mean_difference, 6),
        round(agreement.item() * 100, 2)
    ]
}

for metric, value in zip(
    results["Metric"],
    results["Value"]
):

    print(
        f"{metric}: {value}"
    )

Total Parameters: 201320
FP32 Memory (MB): 0.768
INT8 Memory (MB): 0.192
Theoretical Memory Reduction (%): 75.0
Average Output Difference: 0.003356
Prediction Agreement (%): 80.0


# Conclusion

Model quantization reduces the number of bits used to represent model parameters.

In this experiment, the original model used FP32 parameters while a dynamically quantized version used INT8 representations for supported linear layers.

INT8 representation requires significantly less memory than FP32, making quantization useful for deploying language models on devices with limited memory.

However, quantization can introduce small numerical differences in model outputs. The goal is therefore to achieve lower memory usage while maintaining acceptable model performance.